# Chapter 4 — Coding Exercise
## Representing Data & Engineering Features

Based on **Chapter 4** of *Introduction to Machine Learning with Python*.

## What you'll practice

- **One-hot encode** categorical columns with pandas
- Add **polynomial features** and see the count grow
- **Select features** with univariate and model-based methods

**How to use this notebook:** fill in each cell marked `# TODO`, then run the **Check** cell below it. Full solutions are at the end — try each task yourself first!

> Requires `numpy`, `scikit-learn` (and `pandas` for some chapters).

## Setup

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.feature_selection import SelectPercentile, f_classif, SelectFromModel
from sklearn.ensemble import RandomForestClassifier

# a tiny dataset with categorical columns
df = pd.DataFrame({
    "age":  [25, 45, 33, 52, 29],
    "city": ["NY", "LA", "NY", "SF", "LA"],
    "plan": ["free", "pro", "free", "pro", "free"],
})
df

## Exercise 1 — One-hot encoding

Use `pd.get_dummies(df)` to one-hot encode the categorical columns into `encoded`. How many columns result?

In [ ]:
# TODO
encoded = None

In [ ]:
# Check
assert encoded is not None
print("columns:", list(encoded.columns))
assert any(c.startswith("city_") for c in encoded.columns)
assert any(c.startswith("plan_") for c in encoded.columns)

## Exercise 2 — Polynomial features

On `load_diabetes()` (10 features), apply `PolynomialFeatures(degree=2)`. Print the feature count **before and after**.

In [ ]:
# TODO: print original vs polynomial feature counts
diabetes = load_diabetes()
# YOUR CODE HERE

## Exercise 3 — Univariate feature selection

Add 50 columns of **random noise** to the cancer data, then keep the best 50% of features with `SelectPercentile(percentile=50)`. Compare `LogisticRegression` accuracy **with all** vs **selected** features.

In [ ]:
# TODO: build noisy data, select features, compare accuracy
cancer = load_breast_cancer()
rng = np.random.RandomState(0)
# YOUR CODE HERE

## Exercise 4 — Model-based selection

Use `SelectFromModel(RandomForestClassifier(n_estimators=100, random_state=0), threshold='median')` on the noisy data. How many features does it keep?

In [ ]:
# TODO: fit SelectFromModel and print number of selected features
# YOUR CODE HERE

---
## Solutions

In [ ]:
# Ex1
encoded = pd.get_dummies(df)
print("Ex1 columns:", list(encoded.columns))

# Ex2
diabetes = load_diabetes()
poly = PolynomialFeatures(degree=2).fit(diabetes.data)
Xp = poly.transform(diabetes.data)
print("Ex2 features:", diabetes.data.shape[1], "->", Xp.shape[1])

# Ex3
cancer = load_breast_cancer()
rng = np.random.RandomState(0)
noise = rng.normal(size=(len(cancer.data), 50))
X_noisy = np.hstack([cancer.data, noise])
Xtr, Xte, ytr, yte = train_test_split(X_noisy, cancer.target, random_state=0)
lr_all = LogisticRegression(max_iter=5000).fit(Xtr, ytr).score(Xte, yte)
sel = SelectPercentile(f_classif, percentile=50).fit(Xtr, ytr)
lr_sel = LogisticRegression(max_iter=5000).fit(
    sel.transform(Xtr), ytr).score(sel.transform(Xte), yte)
print(f"Ex3 accuracy all={lr_all:.3f}  selected={lr_sel:.3f}")

# Ex4
sfm = SelectFromModel(
    RandomForestClassifier(n_estimators=100, random_state=0),
    threshold="median").fit(Xtr, ytr)
print("Ex4 selected features:", int(sfm.get_support().sum()), "of", Xtr.shape[1])